In [1]:
import chess, chess.engine, os, stat
from stockfish import Stockfish
from policy import *
import random

In [2]:
engine_path = r".\stockfish\stockfish\stockfish-windows-x86-64-avx2.exe"
sf = Stockfish(engine_path, parameters={"Threads": 4, "Hash": 256})
sf.set_depth(2)           
sf.set_skill_level(2)
sf.get_engine_parameters()

{'Debug Log File': '',
 'Threads': 4,
 'Hash': 256,
 'Ponder': False,
 'MultiPV': 1,
 'Skill Level': 2,
 'Move Overhead': 10,
 'Slow Mover': 100,
 'UCI_Chess960': False,
 'UCI_LimitStrength': False,
 'UCI_Elo': 1350,
 'Contempt': 0,
 'Min Split Depth': 0,
 'Minimum Thinking Time': 20}

In [3]:
games= load_json("data\Bijay_1549_games.json")
agent = Agent("Bijay_1549")
agent.train(games)

<>:1: SyntaxWarning: invalid escape sequence '\B'
<>:1: SyntaxWarning: invalid escape sequence '\B'
C:\Users\yubad\AppData\Local\Temp\ipykernel_7068\871589387.py:1: SyntaxWarning: invalid escape sequence '\B'
  games= load_json("data\Bijay_1549_games.json")


Epoch 1/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 67s 731ms/step - accuracy: 0.0562 - loss: 6.1767 - val_accuracy: 0.1043 - val_loss: 5.6167
Epoch 2/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 94s 871ms/step - accuracy: 0.1452 - loss: 4.5865 - val_accuracy: 0.1755 - val_loss: 4.9688
Epoch 3/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 82s 866ms/step - accuracy: 0.3223 - loss: 2.6223 - val_accuracy: 0.1850 - val_loss: 4.9170
Epoch 4/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 77s 860ms/step - accuracy: 0.5637 - loss: 1.4735 - val_accuracy: 0.2095 - val_loss: 5.1611
Epoch 5/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 83s 877ms/step - accuracy: 0.6866 - loss: 1.0169 - val_accuracy: 0.1937 - val_loss: 5.8453
Epoch 6/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 78s 877ms/step - accuracy: 0.7665 - loss: 0.7442 - val_accuracy: 0.2032 - val_loss: 6.4016
Epoch 7/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 81s 860ms/step - accuracy: 0.8252 - loss: 0.5683 - val_accuracy: 0.2071 - val_loss: 6.5202
Epoch 8/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 77s 861ms/step - accuracy: 0.8598 - loss: 0.4554 - val_accu

In [4]:
board = chess.Board()
def stockfish_move():
    sf.set_fen_position(board.fen())
    move = sf.get_best_move()
    board.push(chess.Move.from_uci(move))
def agent_move():
    move = agent.act(board)
    board.push(move)

In [5]:
NUM_GAMES = 100
import json

games_data = []

for i in range(NUM_GAMES):

    board = chess.Board()
    moves = []

    while not board.is_game_over():

        if board.turn == chess.WHITE:
            move = agent.act(board)

        else:
            sf.set_fen_position(board.fen())
            best = sf.get_best_move()

            if best is None:
                break

            move = chess.Move.from_uci(best)

        board.push(move)
        moves.append(move.uci()) 

    game_data = {
        "event": "Agent vs Stockfish",
        "round": i + 1,
        "white": f"Mimic Agent of {agent.id}",
        "black": "Stockfish",
        "result": board.result(),
        "moves": moves
    }

    games_data.append(game_data)


safe_id = str(agent.id).replace(" ", "_").lower()
with open(f"data/{safe_id}_agent_vs_stockfish.json", "w") as f:
    json.dump(games_data, f, indent=4)